# MIMIC-CXR CheXpert Dataset Extraction

This notebook extracts **500 Pneumonia** cases and **500 random Normal** cases from `mimic-cxr-2.0.0-chexpert.csv` and deletes all other pathology columns.

### Columns Kept:
1. `subject_id` - Patient identifier
2. `study_id` - Study/X-ray session identifier
3. `Pneumonia` - Target finding (1.0 = Pneumonia)
4. `No Finding` - Normal indicator (1.0 = Normal)

In [9]:
import pandas as pd
import numpy as np

## 1. Load the Dataset and Keep Only Selected Columns
We load `mimic-cxr-2.0.0-chexpert.csv` and filter it immediately to keep only `subject_id`, `study_id`, `Pneumonia`, and `No Finding` columns, discarding all other columns.

In [10]:
csv_path = 'mimic-cxr-2.0.0-chexpert.csv'
df = pd.read_csv(csv_path)

# Keep only the required columns
columns_to_keep = ['subject_id', 'study_id', 'Pneumonia', 'No Finding']
df = df[columns_to_keep]

print(f"Loaded and filtered dataset shape: {df.shape}")
print("Remaining columns:", df.columns.tolist())

Loaded and filtered dataset shape: (227827, 4)
Remaining columns: ['subject_id', 'study_id', 'Pneumonia', 'No Finding']


## 2. Inspect the Columns
Let's view the first few rows of the filtered dataset.

In [11]:
df.head()

,subject_id,study_id,Pneumonia,No Finding
0,10000032,50414267,NaN,1.0
1,10000032,53189527,NaN,1.0
2,10000032,53911762,NaN,1.0
3,10000032,56699142,NaN,1.0
4,10000764,57375967,-1.0,NaN


## 3. Extract 500 Pneumonia Cases
We select rows where `Pneumonia == 1.0` and randomly sample 500 of them.

In [12]:
# Filter for Pneumonia == 1.0
pneumonia_cases = df[df['Pneumonia'] == 1.0]
print(f"Total Pneumonia cases available: {len(pneumonia_cases):,}")

# Sample 500 cases
pneumonia_sample = pneumonia_cases.sample(n=500, random_state=42)
print(f"Sampled Pneumonia cases shape: {pneumonia_sample.shape}")

Total Pneumonia cases available: 16,556
Sampled Pneumonia cases shape: (500, 4)


## 4. Extract 500 Random Normal Cases
We select rows where `No Finding == 1.0` and randomly sample 500 of them.

In [13]:
# Filter for No Finding == 1.0
normal_cases = df[df['No Finding'] == 1.0]
print(f"Total Normal (No Finding) cases available: {len(normal_cases):,}")

# Sample 500 cases
normal_sample = normal_cases.sample(n=500, random_state=42)
print(f"Sampled Normal cases shape: {normal_sample.shape}")

Total Normal (No Finding) cases available: 75,455
Sampled Normal cases shape: (500, 4)


## 5. Combine and Shuffle the Samples
We concatenate the two sampled cohorts and shuffle them randomly to ensure balanced mixing.

In [14]:
# Combine samples
combined_df = pd.concat([pneumonia_sample, normal_sample], ignore_index=True)

# Shuffle rows randomly
balanced_df = combined_df.sample(frac=1.0, random_state=42).reset_index(drop=True)
print(f"Combined dataset shape: {balanced_df.shape}")

# Verify target label distribution
print("\nPneumonia label count:")
print(balanced_df['Pneumonia'].value_counts(dropna=False))

print("\nNo Finding label count:")
print(balanced_df['No Finding'].value_counts(dropna=False))

Combined dataset shape: (1000, 4)

Pneumonia label count:
1.0    500
NaN    449
0.0     51
Name: Pneumonia, dtype: int64

No Finding label count:
1.0    500
NaN    500
Name: No Finding, dtype: int64


## 6. Save the Extracted Subset
We save the final balanced 1,000-case dataset into `mimic_subset_1000.csv`.

In [16]:
output_file = 'mimic_subset_1001.csv'
balanced_df.to_csv(output_file, index=False)
print(f"Successfully saved 1,000 clean cases to: {output_file}")

Successfully saved 1,000 clean cases to: mimic_subset_1001.csv
